# Scenario Construction

## 1. Create Dataset Similarity

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from src.Scenario import BenignFlowExtractor, WindowDatasetBuilder, WindowScenarioBuilder

### 1.1 Extract Benign Flows

In [ ]:
extractBenignFlows = False
benignExtractionDirectory = Path(r"E:\CICDDoS2019\01-12\CSV")
benignFlowsOutputPath = Path("data/benign/benign_flows.csv")
benignExtractionChunkSize = 100_000
benignSourceFiles = None
overwriteBenignFlows = False

if extractBenignFlows:
    benignExtraction = BenignFlowExtractor(
        dataDirectory=benignExtractionDirectory,
        chunkSize=benignExtractionChunkSize,
    ).extract(
        outputPath=benignFlowsOutputPath,
        sourceFiles=benignSourceFiles,
        overwrite=overwriteBenignFlows,
    )
    display(pd.DataFrame([benignExtraction]))


### 1.2 Aggregate Existing Benign Flows

In [ ]:
benignFlowsPath = Path("data/benign/benign_flows.csv")
benignAggregationOutputDirectory = Path("data")
benignAggregationChunkSize = 100_000
benignWindowSeconds = [1.0]

benignDatasets = WindowDatasetBuilder(
    dataDirectory=benignFlowsPath.parent,
    outputDirectory=benignAggregationOutputDirectory,
    chunkSize=benignAggregationChunkSize,
).aggregateBenignFlows(
    benignFlowsPath=benignFlowsPath,
    windowSeconds=benignWindowSeconds,
)

display(benignDatasets["summary"])


### 1.3 Aggregate Mixed Attack Flows

In [ ]:
attackDataDirectory = Path(r"E:\CICDDoS2019\01-12\CSV")
attackAggregationOutputDirectory = Path("data")
attackAggregationChunkSize = 100_000
attackWindowSeconds = [1.0]
cutMinutes = None
cutStartMinutes = {}

attackFiles = {'NTP': 'DrDoS_NTP.csv',
    'DNS': 'DrDoS_DNS.csv',
    'LDAP': 'DrDoS_LDAP.csv',
    'MSSQL': 'DrDoS_MSSQL.csv',
    'NETBIOS': 'DrDoS_NetBIOS.csv',
    'SNMP': 'DrDoS_SNMP.csv',
    'SSDP': 'DrDoS_SSDP.csv',
    'UDP': 'DrDoS_UDP.csv',
    'UDP-LAG': 'UDPLag.csv',
    'SYN': 'Syn.csv',
    'TFTP': 'TFTP.csv'}

attackDatasets = WindowDatasetBuilder(
    dataDirectory=attackDataDirectory,
    outputDirectory=attackAggregationOutputDirectory,
    chunkSize=attackAggregationChunkSize,
).aggregateAttackFlows(
    attackFiles=attackFiles,
    windowSeconds=attackWindowSeconds,
    cutMinutes=cutMinutes,
    cutStartMinutes=cutStartMinutes,
)

display(attackDatasets["summary"])


### 1.4 Concatenate Similarity Dataset

In [ ]:
similarityOutputDirectory = Path("data/similarity")
similarityBenignPath = Path("data/benign/benign_1s.csv")
similarityAttacksDirectory = Path("data/attacks/1s")
similaritySequence = ["NTP", "DNS", "LDAP", "MSSQL", "NETBIOS", "SNMP", "SSDP", "UDP", "UDP-LAG", "SYN", "TFTP"]
similarityBlockSeconds = 180
similarityInitialBenignSeconds = 2000
similarityBetweenBenignSeconds = 600
similaritySamplingSeed = 42
similarityAttackPositionsSeconds = None
similarityBenignStartSeconds = 0

similarityDataset = WindowScenarioBuilder(
    benignPath=similarityBenignPath,
    attacksDirectory=similarityAttacksDirectory,
    windowSeconds=1.0,
).build(
    sequence=similaritySequence,
    outputPath=similarityOutputDirectory / f"similarity_1s_{similarityBlockSeconds:g}s.csv",
    blockSeconds=similarityBlockSeconds,
    initialBenignSeconds=similarityInitialBenignSeconds,
    betweenBenignSeconds=similarityBetweenBenignSeconds,
    randomSeed=similaritySamplingSeed,
    attackPositionsSeconds=similarityAttackPositionsSeconds,
    benignStartSeconds=similarityBenignStartSeconds,
    allowBenignReuse=True,
)

display(similarityDataset["summary"])


## 2. Create Controlled Scenarios

In [9]:
from pathlib import Path
from IPython.display import display
from src.Scenario import WindowScenarioBuilder

BenignPath = Path("data/benign/benign_1s.csv")
AttacksDirectory = Path("data/attacks/1s")
OutputDirectory = Path("data/scenario")

BlockSeconds = [60, 120, 240]
InitialBenignSamples = 3600
BetweenBenignSamples = 1200

### 2.1. Consistency

Cria UDP → UDP → UDP com parâmetros próprios, todas as janelas benignas e blocos viáveis de início, meio e fim.


In [10]:
consistencySequence = ['UDP', 'UDP', 'UDP']
consistencySamplingSeed = 42

consistencyResult = WindowScenarioBuilder(
    benignPath=BenignPath,
    attacksDirectory=AttacksDirectory,
    windowSeconds=1.0,
).buildVariants(
    scenarioName="Consistency", sequence=consistencySequence,
    outputDirectory=OutputDirectory, blockSeconds=BlockSeconds,
    initialBenignSeconds=InitialBenignSamples,
    betweenBenignSeconds=BetweenBenignSamples,
    randomSeed=consistencySamplingSeed,
)
display(consistencyResult["availability"])
display(consistencyResult["results"])


,Attack,Occurrences,OriginalSpanSeconds,AvailableWindows,BeginningWindows,MiddleWindows,EndWindows,InternalBenignWindows
0,UDP,3,1669.0,1503,471,476,556,15


,Scenario,WindowSeconds,BlockSeconds,Status,Reason,Path,BlocksPath,Windows,AttackWindows
0,Consistency,1.0,60.0,Gerado,,data\scenario\Consistency\Consistency_1s_60s.csv,data\scenario\Consistency\Consistency_1s_60s_b...,8883,178
1,Consistency,1.0,120.0,Gerado,,data\scenario\Consistency\Consistency_1s_120s.csv,data\scenario\Consistency\Consistency_1s_120s_...,9063,355
2,Consistency,1.0,240.0,Gerado,,data\scenario\Consistency\Consistency_1s_240s.csv,data\scenario\Consistency\Consistency_1s_240s_...,9423,713


### 2.2. Generalization

Cria UDP → SSDP → UDP com parâmetros próprios, todas as janelas benignas e blocos viáveis de início, meio e fim.


In [11]:
generalizationSequence = ['UDP', 'SSDP', 'UDP']
generalizationSamplingSeed = 42

generalizationResult = WindowScenarioBuilder(
    benignPath=BenignPath,
    attacksDirectory=AttacksDirectory,
    windowSeconds=1.0,
).buildVariants(
    scenarioName="Generalization", sequence=generalizationSequence,
    outputDirectory=OutputDirectory, blockSeconds=BlockSeconds,
    initialBenignSeconds=InitialBenignSamples,
    betweenBenignSeconds=BetweenBenignSamples,
    randomSeed=generalizationSamplingSeed,
)
display(generalizationResult["availability"])
display(generalizationResult["results"])


,Attack,Occurrences,OriginalSpanSeconds,AvailableWindows,BeginningWindows,MiddleWindows,EndWindows,InternalBenignWindows
0,UDP,2,1669.0,1503,471,476,556,15
1,SSDP,1,825.0,823,273,275,275,1


,Scenario,WindowSeconds,BlockSeconds,Status,Reason,Path,BlocksPath,Windows,AttackWindows
0,Generalization,1.0,60.0,Gerado,,data\scenario\Generalization\Generalization_1s...,data\scenario\Generalization\Generalization_1s...,8883,177
1,Generalization,1.0,120.0,Gerado,,data\scenario\Generalization\Generalization_1s...,data\scenario\Generalization\Generalization_1s...,9063,356
2,Generalization,1.0,240.0,Gerado,,data\scenario\Generalization\Generalization_1s...,data\scenario\Generalization\Generalization_1s...,9423,713


### 2.3. Adaptation

Cria UDP → SYN → UDP com parâmetros próprios, todas as janelas benignas e blocos viáveis de início, meio e fim.


In [12]:
adaptationSequence = ['UDP', 'SYN', 'UDP']
adaptationSamplingSeed = 42

adaptationResult = WindowScenarioBuilder(
    benignPath=BenignPath,
    attacksDirectory=AttacksDirectory,
    windowSeconds=1.0,
).buildVariants(
    scenarioName="Adaptation", sequence=adaptationSequence,
    outputDirectory=OutputDirectory, blockSeconds=BlockSeconds,
    initialBenignSeconds=InitialBenignSamples,
    betweenBenignSeconds=BetweenBenignSamples,
    randomSeed=adaptationSamplingSeed,
)
display(adaptationResult["availability"])
display(adaptationResult["results"])


,Attack,Occurrences,OriginalSpanSeconds,AvailableWindows,BeginningWindows,MiddleWindows,EndWindows,InternalBenignWindows
0,UDP,2,1669.0,1503,471,476,556,15
1,SYN,1,238.0,238,80,79,79,0


,Scenario,WindowSeconds,BlockSeconds,Status,Reason,Path,BlocksPath,Windows,AttackWindows
0,Adaptation,1.0,60.0,Gerado,,data\scenario\Adaptation\Adaptation_1s_60s.csv,data\scenario\Adaptation\Adaptation_1s_60s_blo...,8883.0,178.0
1,Adaptation,1.0,120.0,Gerado,,data\scenario\Adaptation\Adaptation_1s_120s.csv,data\scenario\Adaptation\Adaptation_1s_120s_bl...,9063.0,357.0
2,Adaptation,1.0,240.0,Inviável,"SYN, estrato meio: necessárias 80 janelas dist...",None,None,NaN,NaN


### 2.4. Recurrence

Cria UDP → SYN → SSDP → UDP → SYN → SSDP com parâmetros próprios, todas as janelas benignas e blocos viáveis de início, meio e fim.


In [ ]:
recurrenceSequence = ['UDP', 'SYN', 'SSDP', 'UDP', 'SYN', 'SSDP']
recurrenceBetweenBenignSamples = 600
recurrenceSamplingSeed = 42

recurrenceResult = WindowScenarioBuilder(
    benignPath=BenignPath,
    attacksDirectory=AttacksDirectory,
    windowSeconds=1.0,
).buildVariants(
    scenarioName="Recurrence", sequence=recurrenceSequence,
    outputDirectory=OutputDirectory, blockSeconds=BlockSeconds,
    initialBenignSeconds=InitialBenignSamples,
    betweenBenignSeconds=recurrenceBetweenBenignSamples,
    randomSeed=recurrenceSamplingSeed,
)
display(recurrenceResult["availability"])
display(recurrenceResult["results"])


,Attack,Occurrences,OriginalSpanSeconds,AvailableWindows,BeginningWindows,MiddleWindows,EndWindows,InternalBenignWindows
0,UDP,2,1669.0,1503,471,476,556,15
1,SYN,2,238.0,238,80,79,79,0
2,SSDP,2,825.0,823,273,275,275,1


,Scenario,WindowSeconds,BlockSeconds,Status,Reason,Path,BlocksPath,Windows,AttackWindows
0,Recurrence,1.0,60.0,Gerado,,data\scenario\Recurrence\Recurrence_1s_60s.csv,data\scenario\Recurrence\Recurrence_1s_60s_blo...,9063.0,357.0
1,Recurrence,1.0,120.0,Inviável,"SYN, estrato meio: necessárias 80 janelas dist...",None,None,NaN,NaN
2,Recurrence,1.0,180.0,Inviável,"SYN, estrato início: necessárias 120 janelas d...",None,None,NaN,NaN
